# Score Variation Under Random Comparison Order

TrueSkill ratings are order dependent: replaying the same comparison history in a
different order produces different scores, even though the comparisons themselves are
identical. This notebook measures how large that effect is on the real dataset.

Each replay is a full pass over **every stored comparison** in a different random
permutation. An image's score moves between replays only because the comparisons it
takes part in landed in a different order relative to its opponents.

Every image in the database is measured, not a sample. Scoring all of them costs
roughly the same wall-clock as a small sample, because the runtime is dominated by the
comparison replay itself rather than by scoring the results. The image and comparison
counts for a given run are printed by the load cell below.

## How to read the axes

Every axis carries its units and a direction note, so no chart needs a legend:

| note | meaning |
|---|---|
| **higher is better** | a quality score; a bigger number is a better-rated image |
| **lower is better (more confident)** | a `sigma_uncertainty` axis; a smaller number means the scorer trusts its own estimate more |
| **lower is better (more stable, not a quality judgement)** | a movement axis (`range` or `std` across replays); a smaller number means the score held still when the order changed, which says nothing about whether the image is good |
| **more comparisons = better-estimated score** | the `comparison_count` axis; more comparisons shrink `sigma_uncertainty` |

That third row is the one worth keeping in mind. Movement is measured on a scale where
**low is good**, but it is a statement about reliability, not about image quality. A highly
rated image and a badly rated image can both be perfectly stable.

## Two score forms, tracked side by side

The module exposes two ways to turn a rating into a number. They are not
interchangeable, and they do not report variation the same way:

| | public score | trueskill score |
|---|---|---|
| formula | win probability vs a default opponent | `mu - 3 * sigma` |
| range | bounded `0.0` to `1.0` | unbounded |
| behaviour at the extremes | saturates, so movement is compressed | stays close to linear |

The public score is a probability, so it flattens out near 0 and 1 and compresses
movement for images that sit near either end. The trueskill score is linear in `mu` and
does not. Section 7 measures the difference. The actual score span for a given run is
printed by the per-image statistics cell.

## A note on the word "std"

There is no "public sigma" in this module. Both score functions above return a single
number, so neither has a spread of its own. Confidence exists only as
`Rating.sigma_uncertainty`, one value per rating.

Every `std` and `range` in this notebook is measured **across the replays** for a single
image, so it describes how much that image's score moves when the comparison order
changes. That is order sensitivity, not confidence, and the two are different things.
Section 4 prints the correlation between them for the current data rather than assuming
a value, so the distinction can be checked instead of trusted.

Nothing here writes to the database. All scores are recalculated in memory, including
the production baseline, which is replayed in stored order rather than read from
`cache.db`.

All tqdm progress bars use `delay=3` matching the codebase style.

In [ ]:
import random
import time

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from tqdm import tqdm as tqdm_desc

from comfyui_image_scorer.domain.analysis.trueskill import (
    public_score_from_rating,
    replay_ratings,
    trueskill_score_from_rating,
)
from comfyui_image_scorer.infrastructure.persistence.comparisons_repository import (
    list_links,
)
from comfyui_image_scorer.infrastructure.persistence.images_repository import list_nodes
from comfyui_image_scorer.infrastructure.persistence.path_handler import find_image_path

from comfyui_image_scorer.core.observability.logger import (
    ModuleLogger,
    configure_package_logging,
    get_logger,
)

# 20 = INFO. __name__ is __main__ in a notebook and configure_package_logging
# sets that logger's level, so anything higher would drop every info record.
configure_package_logging(20)
logger: ModuleLogger = get_logger(__name__)


## 1. Load Data

In [ ]:
image_rows = list_nodes()
comparison_rows = list_links()

filenames: list[str] = [row["filename"] for row in image_rows]
comparison_counts: np.ndarray = np.array(
    [int(row["comparison_count"]) for row in image_rows], dtype=np.int32
)

logger.info(
    "Loaded %d images and %d comparisons",
    len(filenames),
    len(comparison_rows),
)
# replay_ratings only returns images that appear in at least one comparison, and the
# next cell indexes it by filename. A minimum count of 1 is what guarantees that.
logger.info(
    "comparison_count min/median/max: %d / %d / %d",
    int(comparison_counts.min()),
    int(np.median(comparison_counts)),
    int(comparison_counts.max()),
)
logger.info(
    "comparison_count percentiles p50/p90/p99/p99.9: %d / %d / %d / %d",
    int(np.percentile(comparison_counts, 50)),
    int(np.percentile(comparison_counts, 90)),
    int(np.percentile(comparison_counts, 99)),
    int(np.percentile(comparison_counts, 99.9)),
)
# Where the bulk of the data would sit on a full linear axis running to the maximum.
# Whether that is cramped or perfectly fine depends entirely on the shape of the
# distribution, so it is measured here instead of asserted. Section 6 reads these back
# when it picks its own cutoff.
maximum_comparison_count: int = int(comparison_counts.max())
axis_share_at_p90: float = (
    float(np.percentile(comparison_counts, 90)) / maximum_comparison_count
)
logger.info(
    "a full linear axis to the maximum (%d) would place 90%% of images within the "
    "left %.2f%% of its width",
    maximum_comparison_count,
    100.0 * axis_share_at_p90,
)


## 2. Replay Every Image Through Every Order

Builds two `(REPLAY_COUNT, len(filenames))` matrices, one per score form, plus one
production baseline replayed in stored comparison-id order.

In [ ]:
# Number of random permutations of the comparison history replayed per run. A sample
# standard deviation is only pinned to about 1/sqrt(2*(n-1)) relative error, so raise
# this to tighten the per-image spread estimates at the cost of linear time.
REPLAY_COUNT: int = 50

# Seeds the permutation shuffles, the production-order replay, and the axis jitter in
# section 6, so a whole run is reproducible from this one value.
SEED: int = 12300

public_scores = np.empty((REPLAY_COUNT, len(filenames)), dtype=np.float32)
trueskill_scores = np.empty((REPLAY_COUNT, len(filenames)), dtype=np.float32)
# Rating.sigma_uncertainty is kept as its own matrix. The trueskill score is
# mu - 3 * sigma, so folding sigma into one score hides the confidence term
# entirely; section 7 measures it directly.
sigma_uncertainties = np.empty((REPLAY_COUNT, len(filenames)), dtype=np.float32)

replay_start = time.perf_counter()
for replay_index in tqdm_desc(
    range(REPLAY_COUNT), desc="random replays", unit="replay", delay=3
):
    random.seed(SEED + replay_index)
    ratings = replay_ratings(comparison_rows, order="random")
    public_scores[replay_index] = [
        public_score_from_rating(ratings[name][0]) for name in filenames
    ]
    trueskill_scores[replay_index] = [
        trueskill_score_from_rating(ratings[name][0]) for name in filenames
    ]
    sigma_uncertainties[replay_index] = [
        ratings[name][0].sigma_uncertainty for name in filenames
    ]

# The production baseline is replayed through the same path as the random replays
# rather than read from cache.db, so the two are directly comparable.
random.seed(SEED)
default_ratings = replay_ratings(comparison_rows, order="default")
default_public_score = np.array(
    [public_score_from_rating(default_ratings[name][0]) for name in filenames],
    dtype=np.float32,
)
default_trueskill_score = np.array(
    [trueskill_score_from_rating(default_ratings[name][0]) for name in filenames],
    dtype=np.float32,
)
default_sigma_uncertainty = np.array(
    [default_ratings[name][0].sigma_uncertainty for name in filenames],
    dtype=np.float32,
)

logger.info(
    "Replayed %d random orders over %d images (%.1fs)",
    REPLAY_COUNT,
    len(filenames),
    time.perf_counter() - replay_start,
)


## 3. Per-Image Statistics

One row per image: how far its score travels across the replays, and where the
production score sits inside that spread.

In [ ]:
per_image_mean_public = public_scores.mean(axis=0)
per_image_public_std = public_scores.std(axis=0, ddof=1)
per_image_min_public = public_scores.min(axis=0)
per_image_max_public = public_scores.max(axis=0)
per_image_range_public = per_image_max_public - per_image_min_public
per_image_p5_public = np.percentile(public_scores, 5, axis=0)
per_image_p95_public = np.percentile(public_scores, 95, axis=0)

per_image_mean_trueskill = trueskill_scores.mean(axis=0)
per_image_trueskill_std = trueskill_scores.std(axis=0, ddof=1)
per_image_min_trueskill = trueskill_scores.min(axis=0)
per_image_max_trueskill = trueskill_scores.max(axis=0)
per_image_range_trueskill = per_image_max_trueskill - per_image_min_trueskill
per_image_p5_trueskill = np.percentile(trueskill_scores, 5, axis=0)
per_image_p95_trueskill = np.percentile(trueskill_scores, 95, axis=0)

# 0.0 means the production score is below every random replay, 1.0 means above all of
# them, so a flat histogram would mean the production order is just another draw.
default_rank_public = (public_scores < default_public_score).mean(axis=0)

# sigma_uncertainty is the module's own confidence. It is tracked separately from
# both score forms because a score folds it into mu - 3 * sigma.
per_image_uncertainty_mean = sigma_uncertainties.mean(axis=0)
per_image_uncertainty_std = sigma_uncertainties.std(axis=0, ddof=1)
per_image_uncertainty_range = sigma_uncertainties.max(axis=0) - sigma_uncertainties.min(
    axis=0
)

# Highly-ranked images can settle at exactly zero spread across replays, so the
# denominator is masked rather than clipped; those images become NaN and drop out of
# the finite-only stats.
safe_public_std = np.where(per_image_public_std > 0.0, per_image_public_std, np.nan)
default_z_public = (default_public_score - per_image_mean_public) / safe_public_std

logger.info(
    "public score span %.3f to %.3f | trueskill score span %.2f to %.2f "
    "| sigma_uncertainty span %.3f to %.3f",
    float(public_scores.min()),
    float(public_scores.max()),
    float(trueskill_scores.min()),
    float(trueskill_scores.max()),
    float(sigma_uncertainties.min()),
    float(sigma_uncertainties.max()),
)


## 4. Headline Numbers

In [ ]:
summary_table = pd.DataFrame(
    [
        {
            "metric": "public score std",
            "mean": float(per_image_public_std.mean()),
            "median": float(np.median(per_image_public_std)),
            "p90": float(np.percentile(per_image_public_std, 90)),
            "max": float(per_image_public_std.max()),
        },
        {
            "metric": "public range",
            "mean": float(per_image_range_public.mean()),
            "median": float(np.median(per_image_range_public)),
            "p90": float(np.percentile(per_image_range_public, 90)),
            "max": float(per_image_range_public.max()),
        },
        {
            "metric": "trueskill score std",
            "mean": float(per_image_trueskill_std.mean()),
            "median": float(np.median(per_image_trueskill_std)),
            "p90": float(np.percentile(per_image_trueskill_std, 90)),
            "max": float(per_image_trueskill_std.max()),
        },
        {
            "metric": "trueskill range",
            "mean": float(per_image_range_trueskill.mean()),
            "median": float(np.median(per_image_range_trueskill)),
            "p90": float(np.percentile(per_image_range_trueskill, 90)),
            "max": float(per_image_range_trueskill.max()),
        },
    ]
)
display(summary_table.round(4))

# default_rank is a count out of REPLAY_COUNT, so it can only take REPLAY_COUNT + 1
# values. Under pure order noise those are equally likely, so the share that must land
# outside a band comes from counting them. Derived rather than hardcoded so the
# reference stays correct if REPLAY_COUNT is changed.
possible_ranks = np.arange(REPLAY_COUNT + 1) / REPLAY_COUNT
outside_5_95_reference: float = float(
    np.mean((possible_ranks < 0.05) | (possible_ranks > 0.95))
)
outside_1_99_reference: float = float(
    np.mean((possible_ranks < 0.01) | (possible_ranks > 0.99))
)

finite_z = default_z_public[np.isfinite(default_z_public)]
logger.info(
    "production score outside its own random band: [5,95] %.1f%%, [1,99] %.1f%% "
    "(pure order noise would give %.1f%% / %.1f%%)",
    100.0 * float(np.mean((default_rank_public < 0.05) | (default_rank_public > 0.95))),
    100.0 * float(np.mean((default_rank_public < 0.01) | (default_rank_public > 0.99))),
    100.0 * outside_5_95_reference,
    100.0 * outside_1_99_reference,
)
logger.info(
    "z spread of the production score: std %.3f, |z|>2 %.1f%%, |z|>3 %.1f%% "
    "(a normal distribution would give 1.000 / 4.6%% / 0.3%%)",
    float(finite_z.std()),
    100.0 * float(np.mean(np.abs(finite_z) > 2.0)),
    100.0 * float(np.mean(np.abs(finite_z) > 3.0)),
)
logger.info(
    "movement vs confidence: corr(sigma_uncertainty, public std) = %.3f "
    "and (sigma_uncertainty, trueskill std) = %.3f; confidence and order sensitivity "
    "are related but are not the same measurement",
    float(np.corrcoef(default_sigma_uncertainty, per_image_public_std)[0, 1]),
    float(np.corrcoef(default_sigma_uncertainty, per_image_trueskill_std)[0, 1]),
)


## 5. Is the Production Order Just Another Random Draw?

Each image contributes the rank of its production score inside its own distribution of
random-order scores. A rank near `0.0` means the production score is below nearly every
replay; near `1.0`, above nearly every one.

**Reading the dashed line**

The red dashed line is the bar height each bin would have if the comparison order carried
**no information at all** — the image count divided evenly across the bins.

It is an *expectation*, not a pass/fail threshold, so a bar sitting slightly above or
slightly below it means nothing on its own. Every bin has ordinary sampling variation
around the line, and the pale red band marks two standard deviations of it. Inside the
band a bar is unremarkable; outside it is worth a look.

**So is it noise if a bar is over or under the line?**

Not on its own. A rise in one bin is always paid for by a fall in another, so only the
whole shape carries information — never a single bar. That is why the cell prints a
chi-square across all the bins at once, together with how many bars cleared the band. Read
the shape and the chi-square, not the position of any one bar.

Two things the first chart cannot show, so the second pair is there:

- the rank hides the actual score values, so **the two charts below it plot the score
  distributions themselves** — once as the database currently holds them, once as the
  average each image got across the random replays;
- the mean across many replays is close to the score an image would be expected to receive
  under random ordering, so the **gap between those two distributions is the part of the
  score that ordering alone explains**.


In [ ]:
# Bin count for the rank histogram. Ten bins keep each one wide enough to hold a
# readable share of the images.
BIN_COUNT: int = 100
bin_probability: float = 1.0 / BIN_COUNT

rank_histogram, _ = np.histogram(default_rank_public, bins=BIN_COUNT, range=(0.0, 1.0))

# ── What the dashed line means ──────────────────────────────────────────────────
#
# The line is the bar height each bin would have if the comparison order carried no
# information at all: the image count split evenly across the bins. It is an expectation,
# not a pass/fail threshold.
#
# A bar a little above or a little below the line is ordinary sampling variation, so the
# line on its own proves nothing either way. Each bin carries binomial noise of
# sqrt(n * p * (1 - p)) around the expectation, and the shaded band marks +/- 2 of those
# standard deviations. Inside the band a bar is unremarkable; outside it is worth a look.
#
# Only the whole shape can answer the question, because a rise in one bin is always paid
# for by a fall in another. The chi-square below compares all the bars at once.
expected_per_bin: float = len(filenames) / BIN_COUNT
bin_standard_deviation: float = float(
    np.sqrt(len(filenames) * bin_probability * (1.0 - bin_probability))
)
noise_band: float = 2.0 * bin_standard_deviation
chi_squared: float = float(
    ((rank_histogram - expected_per_bin) ** 2 / expected_per_bin).sum()
)
degrees_of_freedom: int = BIN_COUNT - 1
p_value: float = float(stats.chi2.sf(chi_squared, degrees_of_freedom))
bins_outside_band: int = int(
    np.sum(np.abs(rank_histogram - expected_per_bin) > noise_band)
)

figure, axis = plt.subplots(figsize=(12, 5.5))
axis.axhspan(
    expected_per_bin - noise_band,
    expected_per_bin + noise_band,
    color="tab:red",
    alpha=0.10,
    label=f"band of ordinary sampling variation, +/- 2 sd ({noise_band:.0f})",
)
axis.axhline(
    expected_per_bin,
    color="tab:red",
    linestyle="--",
    linewidth=2,
    label=f"expected height if order carried no information ({expected_per_bin:.0f})",
)
axis.bar(
    range(BIN_COUNT),
    rank_histogram,
    color="tab:blue",
    edgecolor="black",
    width=0.82,
    label="observed images",
)
axis.set_xticks(range(BIN_COUNT))
axis.set_xticklabels(
    [
        f"{index / BIN_COUNT:.1f}-{(index + 1) / BIN_COUNT:.1f}"
        for index in range(BIN_COUNT)
    ]
)
axis.set_xlabel(
    "share of random replays that scored below the production score "
    "(fraction, 0-1)\nhigher = the production score sat higher against its own replays"
)
axis.set_ylabel("images (count)")
axis.set_title(
    "Where each production score falls inside its own random spread", fontsize=13
)
axis.grid(axis="y", linestyle=":", alpha=0.4)
axis.legend(loc="upper left", fontsize=9)
figure.tight_layout()
plt.show()

logger.info(
    "per-bin sd under order noise is %.1f, so +/- 2 sd is +/- %.0f; %d of %d bars fall "
    "outside that band; chi2 = %.0f on %d dof, p = %.3g",
    bin_standard_deviation,
    noise_band,
    bins_outside_band,
    BIN_COUNT,
    chi_squared,
    degrees_of_freedom,
    p_value,
)

# ── The same two score distributions, side by side ───────────────────────────────
#
# The rank chart above compresses the scores into a rank. These two charts show the score
# values themselves, scored two ways: as the database currently holds them, and as the
# average each image received across the random replays.
#
# The mean of many replays is close to the score that image would be expected to receive
# under random ordering. So the difference between these two distributions is the part of
# the score that ordering alone can account for.
score_histogram_panels = (
    (default_public_score, "production order, replayed", "tab:blue"),
    (
        per_image_mean_public,
        f"mean of {REPLAY_COUNT} random orders",
        "tab:orange",
    ),
)

figure, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)
for axis, (values, panel_label, color) in zip(axes, score_histogram_panels):
    axis.hist(
        values,
        bins=60,
        range=(0.0, 1.0),
        color=color,
        edgecolor="none",
        alpha=0.85,
    )
    axis.axvline(values.mean(), color="black", linestyle="--", linewidth=2)
    axis.set_title(
        f"{panel_label}\nmean {values.mean():.4f}, sd {values.std():.4f}",
        fontsize=11,
    )
    axis.set_xlabel("public score (0-1 score units, higher is better)")
    axis.grid(axis="y", linestyle=":", alpha=0.4)
axes[0].set_ylabel("images (count)")

figure.suptitle(
    "Score distribution under production order against the average over random orders\n"
    "the gap between these two shapes is the part of the score that ordering explains",
    fontsize=13,
)
figure.tight_layout()
plt.show()


def tail_share(values: np.ndarray) -> float:
    return float(np.mean((values < 0.05) | (values > 0.95)))


logger.info(
    "score spread: production sd %.4f against %.4f for the mean of %d random orders "
    "(ratio %.3f); images beyond 0.05 or 0.95 are %.2f%% against %.2f%%",
    float(default_public_score.std()),
    float(per_image_mean_public.std()),
    REPLAY_COUNT,
    float(default_public_score.std() / per_image_mean_public.std()),
    100.0 * tail_share(default_public_score),
    100.0 * tail_share(per_image_mean_public),
)


## 6. Does Movement Depend On How Many Comparisons An Image Has?

**What is on each axis**

- **x: how many comparisons the image has taken part in.** A count straight from the
  database, so 5 means it was compared five times.
- **y: how much the score moved.** A high dot means that image's score is unstable and
  depends on the order the comparisons arrived in. A low dot means it barely cares. This
  is a measure of movement across the random orders, not a confidence measure.

Two ways of measuring that movement are shown side by side, because they answer slightly
different questions:

- **range** (`max - min`) is the full span the score covered. It answers *how far could
  this score have ended up anywhere?*
- **std** is the typical wobble around its own average. It answers *how much does it
  usually move?* and is far less sensitive to one unlucky replay.

So the chart answers one question: **do images with more comparisons move less?**

**How to read it**

- Every image is its own dot. Nothing is grouped or averaged except the black median
  line, which exists only because the full cloud needs a reference. Bands holding fewer
  than `MIN_BAND_IMAGES` images are skipped, so the line stops where the data stops
  supporting it rather than being drawn through noise.
- `comparison_count` is a whole number, so each dot is nudged sideways by up to half a
  step to stop thousands of dots stacking into a single vertical stripe.
- The x axis is **linear** and stops at `MAX_TREND_COMPARISON_COUNT`. Whether that cutoff
  is needed at all depends on the shape of the data: the load cell measures where the
  bulk of the images would land on a full linear axis, and if that share comes out large
  for your data then this can be raised to the full maximum without crowding anything.
  The chart also logs how many images fall beyond the cutoff and how much they move, so
  the cutoff never hides anything silently.
- Each column keeps its own y axis and carries its own units, because the two score forms
  are on different scales. **Do not compare panel heights across columns or rows**; only
  the shape within a panel is meaningful.


In [ ]:
# The comparison_count axis stops here so the scale can stay linear and readable. The
# load cell measured where the bulk of the data would fall on a full linear axis; if
# that share is large for your data then no cutoff is needed and this can be raised to
# the full maximum without crowding anything.
MAX_TREND_COMPARISON_COUNT: int = 25

# The median line is a reading aid over thousands of dots. Bands thinner than this hold
# too few images for their median to mean anything, so the line stops instead of
# continuing through noise.
MIN_BAND_IMAGES: int = 50

# Even-width bands along the linear axis. Counts are integers, so bands are whole
# numbers wide and a point can never land between two bands.
band_edges = np.unique(np.linspace(1, MAX_TREND_COMPARISON_COUNT, 21).astype(int))
in_axis_range = comparison_counts <= MAX_TREND_COMPARISON_COUNT

# comparison_count is an integer, so every image would otherwise snap onto its own
# vertical stripe and the cloud would read as a picket fence. Spreading each x by
# +/- 0.5 dissolves the stripes without moving any point off its true count.
jitter = np.random.default_rng(SEED).uniform(-0.5, 0.5, size=len(filenames))
jittered_counts = comparison_counts + jitter

# Rows are the two ways of describing movement (full span, and typical wobble);
# columns are the two score forms. Each column keeps its own y axis because the score
# forms carry different units and must not be compared by height.
measures = (
    (
        per_image_range_public,
        "public score range (max - min)",
        "score units on the 0-1 public scale",
        "tab:blue",
    ),
    (
        per_image_public_std,
        "public score std (typical wobble)",
        "score units on the 0-1 public scale",
        "tab:blue",
    ),
    (
        per_image_range_trueskill,
        "trueskill score range (max - min)",
        "trueskill points (mu - 3 sigma)",
        "tab:orange",
    ),
    (
        per_image_trueskill_std,
        "trueskill score std (typical wobble)",
        "trueskill points (mu - 3 sigma)",
        "tab:orange",
    ),
)

# Direction wording shared by every panel in this figure. A movement measure is not a
# quality score: a low value means the score held still across orders, which is a
# statement about reliability rather than about the image itself.
STABLE_DIRECTION = "lower is better (more stable, not a quality judgement)"

figure, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True)
for axis, (values, measure_label, unit_label, color) in zip(axes.flat, measures):
    axis.scatter(
        jittered_counts[in_axis_range],
        values[in_axis_range],
        s=4,
        alpha=0.12,
        color=color,
        edgecolors="none",
    )
    band_centers: list[float] = []
    band_medians: list[float] = []
    for low, high in zip(band_edges[:-1], band_edges[1:]):
        in_band = (
            in_axis_range & (comparison_counts >= low) & (comparison_counts < high)
        )
        if np.count_nonzero(in_band) >= MIN_BAND_IMAGES:
            band_centers.append(float(np.median(comparison_counts[in_band])))
            band_medians.append(float(np.median(values[in_band])))
    axis.plot(band_centers, band_medians, color="black", linewidth=2.5, label="median")
    axis.set_xlim(0, MAX_TREND_COMPARISON_COUNT + 1)
    axis.set_xlabel("comparisons on the image (count; more comparisons = better-estimated score)")
    axis.set_ylabel(unit_label)
    axis.set_title(f"{measure_label}\n{STABLE_DIRECTION}", fontsize=11)
    axis.grid(linestyle=":", alpha=0.4)
    axis.legend(loc="upper right")

figure.suptitle(
    "How much a score moves when the comparison order is shuffled\n"
    f"one dot per image, measured over {REPLAY_COUNT} random orders; "
    "left column public score, right column trueskill score",
    fontsize=13,
)
figure.tight_layout()
plt.show()

beyond_axis = comparison_counts > MAX_TREND_COMPARISON_COUNT
logger.info(
    "%d images (%0.1f%%) sit beyond the axis because they have more than %d "
    "comparisons; their median movement is public range %.4f / std %.4f and "
    "trueskill range %.3f / std %.3f",
    int(beyond_axis.sum()),
    100.0 * float(beyond_axis.mean()),
    MAX_TREND_COMPARISON_COUNT,
    float(np.median(per_image_range_public[beyond_axis])),
    float(np.median(per_image_public_std[beyond_axis])),
    float(np.median(per_image_range_trueskill[beyond_axis])),
    float(np.median(per_image_trueskill_std[beyond_axis])),
)


## 7. sigma_uncertainty, Measured On Its Own

Everything above is a **score**. This section is about `Rating.sigma_uncertainty`, the
module's own confidence, which is a different quantity and is not visible in either score.

It is easy to miss, because `trueskill_score_from_rating` is `mu - 3 * sigma`: the score
bakes sigma in and discards it. Two images with identical `mu` but very different confidence
get different trueskill scores, so part of that score's movement is really sigma moving.

**What is on each axis**

- **x: how many comparisons the image has taken part in** (count).
- **Left panel, y: average sigma** across the random orders, in TrueSkill points. A low
  value means the scorer is confident about that image. This is the confidence curve: more
  comparisons should mean lower sigma.
- **Right panel, y: how much sigma itself moved** across those orders, as a standard
  deviation in TrueSkill points. This answers a different question: not *is the scorer
  confident* but *does the scorer's confidence depend on the order the comparisons arrived
  in*. An image whose sigma is stable across orders sits at zero regardless of how
  confident the scorer is overall.

Both panels plot one dot per image with a black median line, on the same terms as section 6.

**The second figure asks whether reordering changes confidence.** Each row is one
comparison-count band, and every sigma value is centred on its own band's average taken
across *both* orders, so zero is that band's average and the two markers are directly
comparable:

- the pale points are **individual orderings** of sampled images in that band, with a
  small vertical nudge so the density stays visible;
- the **red diamond** is the production order;
- the **black dash** is the mean of the random orders.

The distance between the two markers is the *systematic* effect, which the bar chart on the
right quantifies per band. The width of the point cloud is the *per-ordering* variation.

Keep those two apart. Individual sigma values clearly move — but a movement that averages
out is not the same as confidence getting better or worse.


In [ ]:
# Same cutoff and band rules as section 6, so the two sections can be read side by side.
MAX_TREND_COMPARISON_COUNT: int = 25
MIN_BAND_IMAGES: int = 50

band_edges = np.unique(np.linspace(1, MAX_TREND_COMPARISON_COUNT, 21).astype(int))
in_axis_range = comparison_counts <= MAX_TREND_COMPARISON_COUNT

# Reuse the seed so the sideways nudge matches section 6 dot for dot.
jitter = np.random.default_rng(SEED).uniform(-0.5, 0.5, size=len(filenames))
jittered_counts = comparison_counts + jitter


def median_line(axis, values):
    band_centers: list[float] = []
    band_medians: list[float] = []
    for low, high in zip(band_edges[:-1], band_edges[1:]):
        in_band = (
            in_axis_range & (comparison_counts >= low) & (comparison_counts < high)
        )
        if np.count_nonzero(in_band) >= MIN_BAND_IMAGES:
            band_centers.append(float(np.median(comparison_counts[in_band])))
            band_medians.append(float(np.median(values[in_band])))
    axis.plot(band_centers, band_medians, color="black", linewidth=2.5, label="median")
    axis.set_xlim(0, MAX_TREND_COMPARISON_COUNT + 1)
    axis.set_xlabel("comparisons on the image (count; more comparisons = better-estimated score)")
    axis.grid(linestyle=":", alpha=0.4)
    axis.legend(loc="upper right")


uncertainty_panels = (
    (
        per_image_uncertainty_mean,
        "average sigma_uncertainty",
        "TrueSkill points",
        "lower is better (more confident)",
    ),
    (
        per_image_uncertainty_std,
        "sigma_uncertainty std across replays",
        "TrueSkill points",
        "lower is better (more stable, not a quality judgement)",
    ),
)

figure, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True)
for axis, (values, measure_label, unit_label, direction) in zip(
    axes, uncertainty_panels
):
    axis.scatter(
        jittered_counts[in_axis_range],
        values[in_axis_range],
        s=4,
        alpha=0.12,
        color="tab:purple",
        edgecolors="none",
    )
    median_line(axis, values)
    axis.set_ylabel(unit_label)
    axis.set_title(f"{measure_label}\n{direction}", fontsize=12)

figure.suptitle(
    "Rating.sigma_uncertainty on its own, one dot per image\n"
    f"measured over {REPLAY_COUNT} random comparison orders",
    fontsize=13,
)
figure.tight_layout()
plt.show()

# The trueskill score is mu - 3 * sigma, so sigma moving contributes directly to that
# score's movement. This is the size of that contribution rather than an assumed share.
sigma_contribution = 3.0 * per_image_uncertainty_range
logger.info(
    "sigma's contribution to trueskill-score movement: median 3 * sigma range is %.3f "
    "against a median trueskill-score range of %.3f, so roughly %.0f%% of that score's "
    "movement is the confidence term rather than skill",
    float(np.median(sigma_contribution)),
    float(np.median(per_image_range_trueskill)),
    100.0
    * float(
        np.median(sigma_contribution / np.maximum(per_image_range_trueskill, 1e-9))
    ),
)

beyond_axis = comparison_counts > MAX_TREND_COMPARISON_COUNT
logger.info(
    "%d images (%0.1f%%) sit beyond the axis; their average sigma is %.3f and their "
    "sigma std is %.4f",
    int(beyond_axis.sum()),
    100.0 * float(beyond_axis.mean()),
    float(np.median(per_image_uncertainty_mean[beyond_axis])),
    float(np.median(per_image_uncertainty_std[beyond_axis])),
)

# ── Does random ordering make the scorer more or less confident? ─────────────────
#
# sigma is a conservative quantity, so reordering could leave it alone on average while
# individual images still move. Each row below is one comparison-count band.
#
# Every sigma value is centred on its own band's mean taken across both orders, so zero
# is that band's average and the production marker and the random mean sit on one scale.
# Points are individual orderings; the distance between the two markers is the systematic
# part, while the width of the cloud is the per-ordering variation.
COMPARISON_BAND_EDGES: np.ndarray = np.unique(
    np.linspace(1, MAX_TREND_COMPARISON_COUNT, 11).astype(int)
)
BAND_IMAGE_SAMPLE: int = 200


def band_label(low: int, high: int) -> str:
    """Render a half-open [low, high) band of integer comparison counts."""
    return str(low) if high - 1 == low else f"{low}-{high - 1}"


band_rows: list[tuple[str, np.ndarray]] = []
for low, high in zip(
    COMPARISON_BAND_EDGES[:-1], COMPARISON_BAND_EDGES[1:]
):
    in_band = in_axis_range & (comparison_counts >= low) & (comparison_counts < high)
    if np.count_nonzero(in_band) >= MIN_BAND_IMAGES:
        band_rows.append((band_label(int(low), int(high)), in_band))

# Deterministic image sample per band. Every ordering of every sampled image is kept,
# because the spread of those points is the question.
sampler = np.random.default_rng(SEED)
band_summaries: list[dict[str, object]] = []
for row, (label, in_band) in enumerate(band_rows):
    band_default = default_sigma_uncertainty[in_band]
    band_random = sigma_uncertainties[:, in_band]
    band_centre = float(np.concatenate([band_default, band_random.ravel()]).mean())

    band_images = np.flatnonzero(in_band)
    if band_images.size > BAND_IMAGE_SAMPLE:
        band_images = np.sort(
            sampler.choice(band_images, size=BAND_IMAGE_SAMPLE, replace=False)
        )
    sampled_random = sigma_uncertainties[:, band_images]
    band_summaries.append(
        {
            "row": row,
            "label": label,
            "population": int(np.count_nonzero(in_band)),
            "default_deviation": float(band_default.mean()) - band_centre,
            "random_deviation": float(band_random.mean()) - band_centre,
            "gap": float(band_random.mean()) - float(band_default.mean()),
            "deviations": (sampled_random - band_centre).ravel(),
            "spread": float(sampled_random.std(axis=0, ddof=1).mean()),
            "level": float(band_random.mean()),
        }
    )

# A small vertical nudge keeps the points from collapsing into flat streaks, so the
# density along x stays visible. It moves nothing in the x direction.
spread_limit = float(
    np.percentile(
        np.concatenate([summary["deviations"] for summary in band_summaries]), 99
    )
)
spread_limit = max(spread_limit, 1e-6)

figure, (strip_axis, gap_axis) = plt.subplots(
    1, 2, figsize=(16, 5.5), gridspec_kw={"width_ratios": [3, 2]}
)

for summary in band_summaries:
    row_index = int(summary["row"])
    row = float(row_index)
    deviations = np.asarray(summary["deviations"])
    vertical = row + np.random.default_rng(SEED + row_index).uniform(
        -0.28, 0.28, size=deviations.size
    )
    strip_axis.scatter(
        deviations,
        vertical,
        s=2,
        alpha=0.03,
        color="tab:purple",
        edgecolors="none",
    )
    strip_axis.plot(
        [float(summary["default_deviation"])],
        [row],
        marker="D",
        markersize=7,
        color="tab:red",
        linestyle="none",
        label="production order" if row_index == 0 else None,
    )
    strip_axis.plot(
        [float(summary["random_deviation"])],
        [row],
        marker="_",
        markersize=15,
        markeredgewidth=2.5,
        color="black",
        linestyle="none",
        label="mean of random orders" if row_index == 0 else None,
    )

strip_axis.axvline(0.0, color="grey", linewidth=1.5, linestyle=":")
strip_axis.set_yticks([float(summary["row"]) for summary in band_summaries])
strip_axis.set_yticklabels(
    [
        f"{summary['label']}  (n={summary['population']:,})"
        for summary in band_summaries
    ]
)
# Lowest comparison count at the top, matching how the other charts read left to right.
strip_axis.set_ylim(len(band_summaries) - 0.5, -0.5)
# Clipped to the bulk so a handful of extreme orderings cannot flatten the markers onto
# the zero line, which is the comparison this chart exists to show.
strip_axis.set_xlim(-spread_limit, spread_limit)
strip_axis.set_xlabel(
    "sigma_uncertainty deviation from this band's average (TrueSkill points)\n"
    "clipped to the 99th percentile; lower is better (more confident)"
)
strip_axis.set_ylabel("comparisons on the image (band)")
strip_axis.set_title(
    "Every individual ordering, centred on its own band's average", fontsize=12
)
strip_axis.grid(axis="x", linestyle=":", alpha=0.4)
strip_axis.legend(loc="lower right", fontsize=9)

gaps = [float(summary["gap"]) for summary in band_summaries]
positions = np.arange(len(band_summaries))
gap_axis.axhline(0.0, color="grey", linewidth=1.5)
gap_axis.bar(
    positions,
    gaps,
    # orange where random ordering leaves the scorer more confident, blue where less
    color=["tab:orange" if gap < 0 else "tab:blue" for gap in gaps],
    edgecolor="black",
    width=0.7,
)
gap_axis.set_xticks(positions)
gap_axis.set_xticklabels([str(summary["label"]) for summary in band_summaries])
gap_axis.set_xlabel("comparisons on the image (band)")
gap_axis.set_ylabel(
    "random mean minus production mean\n(TrueSkill points, below zero = more confident)"
)
gap_axis.set_title(
    "Systematic effect of random ordering, by band", fontsize=12
)
gap_axis.grid(axis="y", linestyle=":", alpha=0.4)

figure.suptitle(
    "Does reordering the comparisons make the scorer more or less confident?", fontsize=13
)
figure.tight_layout()
plt.show()

overall_default = float(default_sigma_uncertainty.mean())
overall_random = float(sigma_uncertainties.mean())
relative_shift = 100.0 * (overall_random - overall_default) / overall_default
bands_down = sum(1 for gap in gaps if gap < 0)
relative_spread = float(
    np.median(
        [
            float(summary["spread"]) / float(summary["level"])
            for summary in band_summaries
        ]
    )
)
logger.info(
    "averaged over every image and every order: production sigma %.4f, random sigma "
    "%.4f, so random ordering shifts the average by %+.4f points (%+.1f%%)",
    overall_default,
    overall_random,
    overall_random - overall_default,
    relative_shift,
)
logger.info(
    "the shift has no consistent direction: %d of %d bands move down and %d move up. "
    "Ordering does change individual sigma values, by about %.1f%% of the sigma level "
    "per image, but it does not move the population average in one direction",
    bands_down,
    len(gaps),
    len(gaps) - bands_down,
    100.0 * relative_spread,
)


## 8. Where The Two Scores Disagree

Same images, same x axis, only the y variable differs. Because both panels plot the same
images in the same positions, any difference between the two curves comes from the score
transform itself rather than from which images happen to sit where.

The public score is a probability, so it flattens near 0 and 1: movement should compress
at both ends, making those images look order-insensitive. The trueskill score is linear in
`mu`, so it should stay closer to level. Whether that is what the current data shows is
left for the reader to judge from the two curves.

The correlation in each title measures how strongly a score's distance from the middle
suppresses its own measured movement. A value near `-1` means strong compression, near
`0` means the score form does not care about position.

In [ ]:
# The median line is a reading aid; score bands thinner than this hold too few
# images for their median to mean anything.
MIN_SCORE_BAND_IMAGES: int = 50

distance_from_middle = np.abs(per_image_mean_public - 0.5)

figure, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharex=True)
variance_panels = (
    (
        axes[0],
        per_image_public_std,
        "public score std",
        "score units on the 0-1 public scale",
        "tab:blue",
    ),
    (
        axes[1],
        per_image_trueskill_std,
        "trueskill score std",
        "trueskill points (mu - 3 sigma)",
        "tab:orange",
    ),
)

score_band_edges = np.linspace(0.0, 1.0, 21)
for axis, stds, measure_label, unit_label, color in variance_panels:
    axis.plot(
        per_image_mean_public,
        stds,
        linestyle="none",
        marker=".",
        markersize=2,
        alpha=0.03,
        color=color,
    )
    band_centers: list[float] = []
    band_medians: list[float] = []
    for low, high in zip(score_band_edges[:-1], score_band_edges[1:]):
        in_band = (per_image_mean_public >= low) & (per_image_mean_public < high)
        if np.count_nonzero(in_band) >= MIN_SCORE_BAND_IMAGES:
            band_centers.append(float(np.median(per_image_mean_public[in_band])))
            band_medians.append(float(np.median(stds[in_band])))
    axis.plot(band_centers, band_medians, color="black", linewidth=2.5)
    axis.set_xlabel(
    "mean public score across replays (0-1 public score units, "
    "higher is better)"
)
    axis.set_ylabel(f"{unit_label}\nlower is better (more stable)")
    axis.set_title(
        f"{measure_label}\n"
        f"corr(distance from 0.5, std) = "
        f"{np.corrcoef(distance_from_middle, stds)[0, 1]:.2f}",
        fontsize=12,
    )
    axis.grid(linestyle=":", alpha=0.4)

figure.suptitle(
    "Movement against where the image sits on the score scale\n"
    "a flatter line means the score form compresses movement at that position",
    fontsize=13,
)
figure.tight_layout()
plt.show()


## 9. Distributions

Each panel is labelled with its own units, since the two metrics are not comparable by
eye. The dashed lines mark the population means.

In [ ]:
histogram_panels = (
    (
        per_image_public_std,
        "public score std",
        "score units (0-1 public scale)",
        "tab:blue",
    ),
    (
        per_image_range_public,
        "public range (max - min)",
        "score units (0-1 public scale)",
        "tab:blue",
    ),
    (
        per_image_trueskill_std,
        "trueskill score std",
        "trueskill points (mu - 3 sigma)",
        "tab:orange",
    ),
    (
        per_image_range_trueskill,
        "trueskill range (max - min)",
        "trueskill points (mu - 3 sigma)",
        "tab:orange",
    ),
)

figure, axes = plt.subplots(2, 2, figsize=(14, 8))
for axis, (values, measure_label, unit_label, color) in zip(
    axes.flat, histogram_panels
):
    axis.hist(values, bins=80, color=color, edgecolor="none", alpha=0.85)
    axis.axvline(
        values.mean(), color="black", linestyle="--", linewidth=2, label="mean"
    )
    axis.set_xlabel(f"{measure_label}\n[{unit_label}]\nlower is better (more stable, not a quality judgement)")
    axis.set_ylabel("images (count)")
    axis.grid(axis="y", linestyle=":", alpha=0.4)
    axis.legend(loc="upper right")

figure.suptitle(
    f"How far each of {len(filenames):,} images moves over {REPLAY_COUNT} random orders",
    fontsize=13,
)
figure.tight_layout()
plt.show()


## 10. Example Images Across The Full Score Range

Images are ordered by mean public score and sampled at even positions, so the examples
sit as far apart as the distribution allows and span the whole 0-1 range.

Each title carries both metrics so the thumbnail and the numbers can be read together.
Most of these images have only a handful of comparisons, so they illustrate score spread
rather than image quality.

In [ ]:
# Thumbnails rendered across the public-score range.
EXAMPLE_COUNT: int = 10

by_mean_public = np.argsort(per_image_mean_public)
example_positions = np.linspace(0, len(by_mean_public) - 1, EXAMPLE_COUNT).astype(int)
example_indices = by_mean_public[example_positions]

thumbnail_columns: int = 5
thumbnail_rows: int = int(np.ceil(EXAMPLE_COUNT / thumbnail_columns))

figure, axes = plt.subplots(
    thumbnail_rows,
    thumbnail_columns,
    figsize=(4 * thumbnail_columns, 3.2 * thumbnail_rows),
    squeeze=False,
)
for slot, image_index in enumerate(example_indices):
    axis = axes.flat[slot]
    name = filenames[image_index]
    path = find_image_path(name)
    if path is None:
        axis.text(
            0.5,
            0.5,
            f"not found: {name}",
            ha="center",
            va="center",
            fontsize=8,
        )
    else:
        axis.imshow(mpimg.imread(path))
    axis.set_title(
        f"cc={comparison_counts[image_index]}  "
        f"public {per_image_min_public[image_index]:.3f}-{per_image_max_public[image_index]:.3f}  "
        f"std {per_image_public_std[image_index]:.4f} / "
        f"{per_image_trueskill_std[image_index]:.3f}",
        fontsize=9,
    )
    axis.set_xticks([])
    axis.set_yticks([])

for slot in range(EXAMPLE_COUNT, thumbnail_rows * thumbnail_columns):
    axes.flat[slot].axis("off")

figure.suptitle(
    f"{EXAMPLE_COUNT} images spread across the public-score range "
    "(public std / trueskill std)",
    fontsize=13,
)
figure.tight_layout()
plt.show()


## 11. Most Movement, Least Movement

The same numbers without a chart, for the images that move the most and the ones that
have effectively settled. `default_rank` near 0 or 1 means the production score landed at
the edge of that image's random spread.

In [ ]:
# Rows shown by each table, and the comparison floor for the settled table.
TABLE_ROWS: int = 10
SETTLED_MIN_COMPARISONS: int = 10

movement_table = pd.DataFrame(
    {
        "filename": filenames,
        "comparisons": comparison_counts,
        "public_mean": per_image_mean_public,
        "public_min": per_image_min_public,
        "public_max": per_image_max_public,
        "public_std": per_image_public_std,
        "trueskill_std": per_image_trueskill_std,
        "default_public": default_public_score,
        "default_trueskill": default_trueskill_score,
        "default_rank": default_rank_public,
    }
)

print(f"Most movement (top {TABLE_ROWS} by public score std)")
display(
    movement_table.sort_values("public_std", ascending=False).head(TABLE_ROWS).round(4)
)

print(
    f"Most settled (top {TABLE_ROWS} by public score std, "
    f"at least {SETTLED_MIN_COMPARISONS} comparisons)"
)
display(
    movement_table[movement_table["comparisons"] >= SETTLED_MIN_COMPARISONS]
    .sort_values("public_std")
    .head(TABLE_ROWS)
    .round(4)
)
